# NEX-CIV Phase 3: LightGBM Complaint Priority & Triage Model
## Production Training on 10,000 Real Multilingual Municipal Grievances

This notebook trains the **LightGBM Regressor & Priority Triage Engine** for Andhra Pradesh Urban Local Bodies.
It extracts TF-IDF multilingual lexical representations (Telugu + English) and fuses them with municipal domain features:
- Public Safety Risk
- Public Health & Contamination Risk
- Log-Scaled Population Impact
- Infrastructure Criticality & Arterial Transit
- Statutory SLA Days (AP Act No. 6 of 2018)
- Emergency Override Hazards (Live Wires, Cave-ins, Open Manholes)

In [ ]:
# 1. Install Dependencies in Google Colab
!pip install lightgbm scikit-learn pandas numpy joblib matplotlib seaborn


In [ ]:
# 2. Imports and Environment Setup
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, classification_report
import joblib
import json
from pathlib import Path

print(f"LightGBM Version: {lgb.__version__}")


In [ ]:
# 3. Load the Real AP Grievance Dataset (10,000 complaints)
# In Google Colab: upload expert_civic_grievance_dataset.csv or mount Google Drive
DATASET_FILE = "expert_civic_grievance_dataset.csv"

if not Path(DATASET_FILE).exists():
    for alt in ["../datasets/expert_civic_grievance_dataset.csv", "priority_engine/datasets/expert_civic_grievance_dataset.csv"]:
        if Path(alt).exists():
            DATASET_FILE = alt
            break

df = pd.read_csv(DATASET_FILE)
print(f"Loaded {len(df)} grievances from {DATASET_FILE}")
df[["complaint_id", "text", "category", "safety_risk", "priority_score"]].head()


In [ ]:
# 4. Feature Engineering: NLP TF-IDF + Tabular Civic Features
df["text"] = df["text"].fillna("")
df["category"] = df["category"].fillna("road")
df["safety_risk"] = pd.to_numeric(df["safety_risk"], errors="coerce").fillna(50.0)
df["public_health_risk"] = pd.to_numeric(df["public_health_risk"], errors="coerce").fillna(50.0)
df["estimated_citizens_affected"] = pd.to_numeric(df["estimated_citizens_affected"], errors="coerce").fillna(50.0)
df["infrastructure_criticality"] = pd.to_numeric(df["infrastructure_criticality"], errors="coerce").fillna(50.0)
df["statutory_sla_days"] = pd.to_numeric(df["statutory_sla_days"], errors="coerce").fillna(3.0)
df["is_emergency"] = df["is_emergency"].astype(int)
df["priority_score"] = pd.to_numeric(df["priority_score"], errors="coerce").fillna(50.0)

tfidf = TfidfVectorizer(max_features=500, ngram_range=(1, 2), token_pattern=r"(?u)\b\w+\b")
text_features = tfidf.fit_transform(df["text"]).toarray()

cat_dummies = pd.get_dummies(df["category"], prefix="cat", drop_first=False)
cat_feature_names = list(cat_dummies.columns)
log_citizens = np.log1p(df["estimated_citizens_affected"].values).reshape(-1, 1)

tabular_features = np.column_stack([
    df["safety_risk"].values,
    df["public_health_risk"].values,
    log_citizens,
    df["infrastructure_criticality"].values,
    df["statutory_sla_days"].values,
    df["is_emergency"].values,
    cat_dummies.values
])

X = np.hstack([tabular_features, text_features])
y = df["priority_score"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")


In [ ]:
# 5. Train LightGBM Regressor with Regularization
lgb_model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=6,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42,
    importance_type="gain",
    verbose=-1
)

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)]
)


In [ ]:
# 6. Evaluate Model Performance
y_pred = lgb_model.predict(X_test)
y_pred = np.clip(y_pred, 0.0, 100.0)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f} points (on 0-100 scale)")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} points")
print(f"R² Score: {r2:.4f}")

def to_tier(score):
    if score >= 80: return "Critical"
    if score >= 60: return "High"
    if score >= 40: return "Medium"
    return "Low"

y_test_tiers = [to_tier(s) for s in y_test]
y_pred_tiers = [to_tier(s) for s in y_pred]
print("\nClassification Report by Municipal Priority Bands:")
print(classification_report(y_test_tiers, y_pred_tiers, digits=3))


In [ ]:
# 7. Export the Model Artifact (.pkl) for NEX-CIV Deployment
package = {
    "model": lgb_model,
    "tfidf": tfidf,
    "cat_feature_names": cat_feature_names,
    "mae": float(mae),
    "rmse": float(rmse),
    "r2": float(r2),
    "features_count": X.shape[1]
}
OUTPUT_PATH = "nexciv_lightgbm_priority_model.pkl"
joblib.dump(package, OUTPUT_PATH)
print(f"Model artifact exported to {OUTPUT_PATH}")
# In Google Colab, you can download the file with:
# from google.colab import files; files.download(OUTPUT_PATH)
